# Visualization Design: The Simpsons Episodes

This notebook explores different chart designs for each research question.
We compare multiple approaches to find the most effective visualization.


In [1]:
import altair as alt
import pandas as pd
import numpy as np

In [2]:
# Load clean dataset
df = pd.read_csv("../data/clean_data/simpsons_episodes_clean.csv")
df["original_air_date"] = pd.to_datetime(df["original_air_date"])

# Derived columns
df["era"] = pd.cut(
    df["season"],
    bins=[0, 8, 18, 27],
    labels=["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"],
)
np.random.seed(42)
df["jitter_season"] = df["season"] + np.random.uniform(-0.25, 0.25, len(df))

# Season boundaries: vertical lines between last ep of season N and first ep of season N+1
season_bounds = (
    df.groupby("season")["number_in_series"].agg(["min", "max"]).reset_index()
)
season_dividers = pd.DataFrame(
    {
        "x": [
            (season_bounds.loc[i, "max"] + season_bounds.loc[i + 1, "min"]) / 2
            for i in range(len(season_bounds) - 1)
        ],
        "season_label": [str(s) for s in season_bounds["season"].iloc[:-1]],
    }
)

print(f"Shape: {df.shape}")
print(f"\nEra distribution:")
print(df["era"].value_counts().sort_index())
df.head()

Shape: (596, 11)

Era distribution:
era
Golden Age (S1-8)    178
Middle (S9-18)       222
Later (S19-27)       196
Name: count, dtype: int64


,title,season,number_in_season,number_in_series,original_air_date,weekday,imdb_rating,imdb_votes,us_viewers_in_millions,era,jitter_season
0,Simpsons Roasting on an Open Fire,1,1,1,1989-12-17,Sunday,8.2,3734.0,26.7,Golden Age (S1-8),0.937270
1,Bart the Genius,1,2,2,1990-01-14,Sunday,7.8,1973.0,24.5,Golden Age (S1-8),1.225357
2,Homer's Odyssey,1,3,3,1990-01-21,Sunday,7.5,1709.0,27.5,Golden Age (S1-8),1.115997
3,There's No Disgrace Like Home,1,4,4,1990-01-28,Sunday,7.8,1701.0,20.2,Golden Age (S1-8),1.049329
4,Bart the General,1,5,5,1990-02-04,Sunday,8.1,1732.0,27.1,Golden Age (S1-8),0.828009


In [3]:
# Reusable layer: season divider lines + labels
# x-axis config that hides vertical grid (use in all episode-based scatter charts)
episode_x = lambda title="Episode Number": alt.X(
    "number_in_series:Q",
    title=title,
    axis=alt.Axis(grid=False),
)

season_rules = (
    alt.Chart(season_dividers)
    .mark_rule(color="gray", strokeDash=[4, 4], opacity=0.4)
    .encode(x=alt.X("x:Q", axis=alt.Axis(grid=False)))
)

# Labels positioned at the midpoint of each season
season_midpoints = (
    df.groupby("season")["number_in_series"].agg(["min", "max"]).reset_index()
)
season_midpoints["mid"] = (season_midpoints["min"] + season_midpoints["max"]) / 2
season_midpoints["label"] = season_midpoints["season"].astype(str)


def season_labels(y_value):
    """Create season number labels at a given y pixel position."""
    return (
        alt.Chart(season_midpoints)
        .mark_text(fontSize=8, color="gray", dy=-5)
        .encode(
            x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
            y=alt.value(y_value),
            text="label:N",
        )
    )


# Era boundaries and labels (for thick era divider lines)
era_bounds = df.groupby("era")["number_in_series"].agg(["min", "max"]).reset_index()
era_bounds["mid"] = (era_bounds["min"] + era_bounds["max"]) / 2

# Divider lines between eras (between last ep of era N and first ep of era N+1)
era_divider_x = [
    (era_bounds.loc[i, "max"] + era_bounds.loc[i + 1, "min"]) / 2
    for i in range(len(era_bounds) - 1)
]
era_divider_data = pd.DataFrame({"x": era_divider_x})

era_rules = (
    alt.Chart(era_divider_data)
    .mark_rule(color="black", strokeWidth=2, opacity=0.6)
    .encode(x=alt.X("x:Q", axis=alt.Axis(grid=False)))
)


def era_labels(y_value=-15):
    """Create era name labels above the chart (negative y = above)."""
    return (
        alt.Chart(era_bounds)
        .mark_text(fontSize=10, fontWeight="bold", color="black")
        .encode(
            x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
            y=alt.value(y_value),
            text="era:N",
        )
    )

## How have the ratings evolved over time?

We compare four approaches: box plot with mean overlay, and three scatter-based variants with different trend lines (LOESS, rolling mean with std band, season mean step line).


In [4]:
# Box plot per season + mean line overlay
# Pre-compute season mean for coloring (inline aggregate breaks outlier points)
season_mean_rating = (
    df.groupby("season")["imdb_rating"].mean().rename("season_mean_rating")
)
df_q1 = df.merge(season_mean_rating, on="season")

q1c_box = (
    alt.Chart(df_q1)
    .mark_boxplot(size=15)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.Color(
            "season_mean_rating:Q",
            scale=alt.Scale(scheme="blues"),
            legend=None,
        ),
    )
)

q1c_mean = (
    alt.Chart(df_q1)
    .mark_line(color="black", strokeWidth=2)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("mean(imdb_rating):Q"),
    )
)

q1c_points = q1c_mean.mark_point(color="black", filled=True, size=40)

(q1c_box + q1c_mean + q1c_points).properties(
    width=700, height=350, title="IMDb Rating per Season"
)

alt.LayerChart(...)

In [5]:
# Box plot per season + mean line (colored by era)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q1c_era_box = (
    alt.Chart(df)
    .mark_boxplot(size=15)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=alt.Legend(title="Era"),
        ),
    )
)

q1c_era_mean = (
    alt.Chart(df)
    .mark_line(color="black", strokeWidth=2)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("mean(imdb_rating):Q"),
    )
)

q1c_era_points = q1c_era_mean.mark_point(color="black", filled=True, size=40)

(q1c_era_box + q1c_era_mean + q1c_era_points).properties(
    width=700, height=350, title="IMDb Rating per Season (Era Colors)"
)

alt.LayerChart(...)

In [6]:
# Scatter per episode + LOESS trend line (with season dividers)
q1_loess_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=episode_x(),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_loess_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "imdb_rating", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("imdb_rating:Q"),
    )
)

(q1_loess_dots + q1_loess_line + season_rules + season_labels(10)).properties(
    width=700,
    height=350,
    title="IMDb Rating per Episode (LOESS, episode axis + season dividers)",
)

alt.LayerChart(...)

In [7]:
# LOESS with season numbers on x-axis (replacing episode numbers)
q1_loess_s_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X(
            "number_in_series:Q",
            title="Season",
            axis=alt.Axis(grid=False, labels=False, ticks=False, titlePadding=20),
        ),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_loess_s_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "imdb_rating", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("imdb_rating:Q"),
    )
)

# Season number labels positioned just below the chart area
q1_season_x_labels = (
    alt.Chart(season_midpoints)
    .mark_text(fontSize=9, color="black")
    .encode(
        x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
        y=alt.value(360),
        text="label:N",
    )
)

(q1_loess_s_dots + q1_loess_s_line + season_rules + q1_season_x_labels).properties(
    width=700,
    height=350,
    title="IMDb Rating per Episode",
)

alt.LayerChart(...)

In [8]:
# LOESS with era dividers and era labels
q1_era_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=episode_x(),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_era_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "imdb_rating", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("imdb_rating:Q"),
    )
)

(q1_era_dots + q1_era_line + era_rules + era_labels()).properties(
    width=700, height=350, title="IMDb Rating per Episode (LOESS + Era Dividers)"
)

alt.LayerChart(...)

In [9]:
# LOESS with season numbers on x-axis + era dividers + season lines
q1_era_s_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X(
            "number_in_series:Q",
            title="Season",
            axis=alt.Axis(grid=False, labels=False, ticks=False, titlePadding=20),
        ),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_era_s_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "imdb_rating", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("imdb_rating:Q"),
    )
)

q1_era_s_xlabels = (
    alt.Chart(season_midpoints)
    .mark_text(fontSize=9, color="black")
    .encode(
        x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
        y=alt.value(360),
        text="label:N",
    )
)

(
    q1_era_s_dots
    + q1_era_s_line
    + season_rules
    + era_rules
    + era_labels()
    + q1_era_s_xlabels
).properties(
    width=700,
    height=350,
    title="IMDb Rating per Episode (LOESS, season axis + era dividers)",
)

alt.LayerChart(...)

In [10]:
# LOESS with season axis + era dividers + season lines + dots colored by era
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q1_erac_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.4)
    .encode(
        x=alt.X(
            "number_in_series:Q",
            title="Season",
            axis=alt.Axis(grid=False, labels=False, ticks=False, titlePadding=20),
        ),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=None,
        ),
        tooltip=["title:N", "season:O", "era:N", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_erac_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "imdb_rating", bandwidth=0.2)
    .mark_line(color="black", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("imdb_rating:Q"),
    )
)

q1_erac_xlabels = (
    alt.Chart(season_midpoints)
    .mark_text(fontSize=9, color="black")
    .encode(
        x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
        y=alt.value(360),
        text="label:N",
    )
)

(
    q1_erac_dots
    + q1_erac_line
    + season_rules
    + era_rules
    + era_labels()
    + q1_erac_xlabels
).properties(
    width=700,
    height=350,
    title="IMDb Rating per Episode (LOESS, season axis + era colors)",
)

alt.LayerChart(...)

In [11]:
# Scatter per episode + rolling mean with confidence band
window = 60

df_sorted = df.sort_values("number_in_series").copy()
df_sorted["rolling_mean_r"] = (
    df_sorted["imdb_rating"].rolling(window, center=True).mean()
)
df_sorted["rolling_std_r"] = df_sorted["imdb_rating"].rolling(window, center=True).std()
df_sorted["rolling_upper_r"] = df_sorted["rolling_mean_r"] + df_sorted["rolling_std_r"]
df_sorted["rolling_lower_r"] = df_sorted["rolling_mean_r"] - df_sorted["rolling_std_r"]

q1_roll_dots = (
    alt.Chart(df_sorted)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X("number_in_series:Q", title="Episode Number"),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_roll_band = (
    alt.Chart(df_sorted)
    .mark_area(opacity=0.2, color="#e45756")
    .encode(
        x=alt.X("number_in_series:Q"),
        y=alt.Y("rolling_lower_r:Q"),
        y2=alt.Y2("rolling_upper_r:Q"),
    )
)

q1_roll_line = (
    alt.Chart(df_sorted)
    .mark_line(color="#e45756", strokeWidth=2.5)
    .encode(
        x=alt.X("number_in_series:Q"),
        y=alt.Y("rolling_mean_r:Q"),
    )
)

(q1_roll_dots + q1_roll_band + q1_roll_line).properties(
    width=700,
    height=350,
    title=f"IMDb Rating per Episode (Rolling Mean +/- 1 Std Dev, window={window})",
)

alt.LayerChart(...)

In [12]:
# Scatter per episode + season mean step line (simplest option)
season_means_r = (
    df.groupby("season")
    .agg(
        mean_rating=("imdb_rating", "mean"),
        ep_start=("number_in_series", "min"),
        ep_end=("number_in_series", "max"),
    )
    .reset_index()
)

steps = pd.concat(
    [
        season_means_r[["season", "mean_rating", "ep_start"]].rename(
            columns={"ep_start": "ep"}
        ),
        season_means_r[["season", "mean_rating", "ep_end"]].rename(
            columns={"ep_end": "ep"}
        ),
    ]
).sort_values(["season", "ep"])

q1_step_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X("number_in_series:Q", title="Episode Number"),
        y=alt.Y(
            "imdb_rating:Q",
            title="IMDb Rating",
            scale=alt.Scale(domain=[4, 10]),
        ),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "number_in_season:O", "imdb_rating:Q"],
    )
)

q1_step_line = (
    alt.Chart(steps)
    .mark_line(color="#e45756", strokeWidth=2.5)
    .encode(
        x=alt.X("ep:Q"),
        y=alt.Y("mean_rating:Q"),
        detail="season:N",
    )
)

q1_step_pts = (
    alt.Chart(season_means_r)
    .mark_point(color="#e45756", filled=True, size=40)
    .encode(
        x=alt.X("ep_start:Q"),
        y=alt.Y("mean_rating:Q"),
        tooltip=[
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("mean_rating:Q", title="Mean Rating", format=".2f"),
        ],
    )
)

(q1_step_dots + q1_step_line + q1_step_pts).properties(
    width=700, height=350, title="IMDb Rating per Episode (Season Mean Step Line)"
)

alt.LayerChart(...)

## How have the viewers evolved over time?

Same four chart types as above, applied to `us_viewers_in_millions` for visual consistency.


In [13]:
# Box plot per season + mean line overlay
season_mean_viewers = (
    df.groupby("season")["us_viewers_in_millions"].mean().rename("season_mean_viewers")
)
df_q2 = df.merge(season_mean_viewers, on="season")

q2c_box = (
    alt.Chart(df_q2)
    .mark_boxplot(size=15)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "season_mean_viewers:Q",
            scale=alt.Scale(scheme="oranges"),
            legend=None,
        ),
    )
)

q2c_mean = (
    alt.Chart(df_q2)
    .mark_line(color="black", strokeWidth=2)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("mean(us_viewers_in_millions):Q"),
    )
)

q2c_points = q2c_mean.mark_point(color="black", filled=True, size=40)

(q2c_box + q2c_mean + q2c_points).properties(
    width=700, height=350, title="US Viewers per Season"
)

alt.LayerChart(...)

In [14]:
# Box plot per season + mean line (colored by era)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q2c_era_box = (
    alt.Chart(df)
    .mark_boxplot(size=15)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=alt.Legend(title="Era"),
        ),
    )
)

q2c_era_mean = (
    alt.Chart(df)
    .mark_line(color="black", strokeWidth=2)
    .encode(
        x=alt.X("season:O"),
        y=alt.Y("mean(us_viewers_in_millions):Q"),
    )
)

q2c_era_points = q2c_era_mean.mark_point(color="black", filled=True, size=40)

(q2c_era_box + q2c_era_mean + q2c_era_points).properties(
    width=700, height=350, title="US Viewers per Season (Era Colors)"
)

alt.LayerChart(...)

In [15]:
# Scatter per episode + LOESS trend line
q2_loess_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X("number_in_series:Q", title="Episode Number"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#f58518"),
        tooltip=[
            "title:N",
            "season:O",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_loess_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "us_viewers_in_millions", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q"),
        y=alt.Y("us_viewers_in_millions:Q"),
    )
)

(q2_loess_dots + q2_loess_line).properties(
    width=700, height=350, title="US Viewers per Episode (Scatter + LOESS Trend)"
)

alt.LayerChart(...)

In [16]:
# LOESS with season numbers on x-axis (replacing episode numbers)
q2_loess_s_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X(
            "number_in_series:Q",
            title="Season",
            axis=alt.Axis(grid=False, labels=False, ticks=False, titlePadding=20),
        ),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#f58518"),
        tooltip=[
            "title:N",
            "season:O",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_loess_s_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "us_viewers_in_millions", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("us_viewers_in_millions:Q"),
    )
)

q2_season_x_labels = (
    alt.Chart(season_midpoints)
    .mark_text(fontSize=9, color="black")
    .encode(
        x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
        y=alt.value(360),
        text="label:N",
    )
)

(q2_loess_s_dots + q2_loess_s_line + season_rules + q2_season_x_labels).properties(
    width=700, height=350, title="US Viewers per Episode"
)

alt.LayerChart(...)

In [17]:
# LOESS with era dividers and era labels
q2_era_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=episode_x(),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#f58518"),
        tooltip=[
            "title:N",
            "season:O",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_era_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "us_viewers_in_millions", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("us_viewers_in_millions:Q"),
    )
)

(q2_era_dots + q2_era_line + era_rules + era_labels()).properties(
    width=700, height=350, title="US Viewers per Episode (LOESS + Era Dividers)"
)

alt.LayerChart(...)

In [18]:
# LOESS with season numbers on x-axis + era dividers + season lines
q2_era_s_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X(
            "number_in_series:Q",
            title="Season",
            axis=alt.Axis(grid=False, labels=False, ticks=False, titlePadding=20),
        ),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#f58518"),
        tooltip=[
            "title:N",
            "season:O",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_era_s_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "us_viewers_in_millions", bandwidth=0.2)
    .mark_line(color="#e45756", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("us_viewers_in_millions:Q"),
    )
)

q2_era_s_xlabels = (
    alt.Chart(season_midpoints)
    .mark_text(fontSize=9, color="black")
    .encode(
        x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
        y=alt.value(360),
        text="label:N",
    )
)

(
    q2_era_s_dots
    + q2_era_s_line
    + season_rules
    + era_rules
    + era_labels()
    + q2_era_s_xlabels
).properties(
    width=700,
    height=350,
    title="US Viewers per Episode (LOESS, season axis + era dividers)",
)

alt.LayerChart(...)

In [19]:
# LOESS with season axis + era dividers + season lines + dots colored by era
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q2_erac_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.4)
    .encode(
        x=alt.X(
            "number_in_series:Q",
            title="Season",
            axis=alt.Axis(grid=False, labels=False, ticks=False, titlePadding=20),
        ),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            legend=None,
        ),
        tooltip=[
            "title:N",
            "season:O",
            "era:N",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_erac_line = (
    alt.Chart(df)
    .transform_loess("number_in_series", "us_viewers_in_millions", bandwidth=0.2)
    .mark_line(color="black", strokeWidth=3)
    .encode(
        x=alt.X("number_in_series:Q", axis=alt.Axis(grid=False)),
        y=alt.Y("us_viewers_in_millions:Q"),
    )
)

q2_erac_xlabels = (
    alt.Chart(season_midpoints)
    .mark_text(fontSize=9, color="black")
    .encode(
        x=alt.X("mid:Q", axis=alt.Axis(grid=False)),
        y=alt.value(360),
        text="label:N",
    )
)

(
    q2_erac_dots
    + q2_erac_line
    + season_rules
    + era_rules
    + era_labels()
    + q2_erac_xlabels
).properties(
    width=700,
    height=350,
    title="US Viewers per Episode (LOESS, season axis + era colors)",
)

alt.LayerChart(...)

In [20]:
# Scatter per episode + rolling mean with confidence band
window = 30

df_sorted_v = df.sort_values("number_in_series").copy()
df_sorted_v["rolling_mean_v"] = (
    df_sorted_v["us_viewers_in_millions"].rolling(window, center=True).mean()
)
df_sorted_v["rolling_std_v"] = (
    df_sorted_v["us_viewers_in_millions"].rolling(window, center=True).std()
)
df_sorted_v["rolling_upper_v"] = (
    df_sorted_v["rolling_mean_v"] + df_sorted_v["rolling_std_v"]
)
df_sorted_v["rolling_lower_v"] = (
    df_sorted_v["rolling_mean_v"] - df_sorted_v["rolling_std_v"]
)

q2_roll_dots = (
    alt.Chart(df_sorted_v)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X("number_in_series:Q", title="Episode Number"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#f58518"),
        tooltip=[
            "title:N",
            "season:O",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_roll_band = (
    alt.Chart(df_sorted_v)
    .mark_area(opacity=0.2, color="#e45756")
    .encode(
        x=alt.X("number_in_series:Q"),
        y=alt.Y("rolling_lower_v:Q"),
        y2=alt.Y2("rolling_upper_v:Q"),
    )
)

q2_roll_line = (
    alt.Chart(df_sorted_v)
    .mark_line(color="#e45756", strokeWidth=2.5)
    .encode(
        x=alt.X("number_in_series:Q"),
        y=alt.Y("rolling_mean_v:Q"),
    )
)

(q2_roll_dots + q2_roll_band + q2_roll_line).properties(
    width=700,
    height=350,
    title="US Viewers per Episode (Rolling Mean +/- 1 Std Dev, window=30)",
)

alt.LayerChart(...)

In [21]:
# Scatter per episode + season mean step line (simplest option)
season_means_v = (
    df.groupby("season")
    .agg(
        mean_viewers=("us_viewers_in_millions", "mean"),
        ep_start=("number_in_series", "min"),
        ep_end=("number_in_series", "max"),
    )
    .reset_index()
)

steps_v = pd.concat(
    [
        season_means_v[["season", "mean_viewers", "ep_start"]].rename(
            columns={"ep_start": "ep"}
        ),
        season_means_v[["season", "mean_viewers", "ep_end"]].rename(
            columns={"ep_end": "ep"}
        ),
    ]
).sort_values(["season", "ep"])

q2_step_dots = (
    alt.Chart(df)
    .mark_circle(size=20, opacity=0.35)
    .encode(
        x=alt.X("number_in_series:Q", title="Episode Number"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#f58518"),
        tooltip=[
            "title:N",
            "season:O",
            "number_in_season:O",
            "us_viewers_in_millions:Q",
        ],
    )
)

q2_step_line = (
    alt.Chart(steps_v)
    .mark_line(color="#e45756", strokeWidth=2.5)
    .encode(
        x=alt.X("ep:Q"),
        y=alt.Y("mean_viewers:Q"),
        detail="season:N",
    )
)

q2_step_pts = (
    alt.Chart(season_means_v)
    .mark_point(color="#e45756", filled=True, size=40)
    .encode(
        x=alt.X("ep_start:Q"),
        y=alt.Y("mean_viewers:Q"),
        tooltip=[
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("mean_viewers:Q", title="Mean Viewers (M)", format=".2f"),
        ],
    )
)

(q2_step_dots + q2_step_line + q2_step_pts).properties(
    width=700, height=350, title="US Viewers per Episode (Season Mean Step Line)"
)

alt.LayerChart(...)

## Is there a correlation between ratings and viewers?

Four scatter plot variants with different color encodings and trendlines to disentangle the direct correlation from the shared time-based confound.


In [22]:
# Scatter colored by season (all 27)
alt.Chart(df).mark_circle(size=50, opacity=0.6).encode(
    x=alt.X("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
    y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
    color=alt.Color("season:Q", scale=alt.Scale(scheme="turbo"), title="Season"),
    tooltip=["title:N", "season:O", "imdb_rating:Q", "us_viewers_in_millions:Q"],
).properties(width=500, height=400, title="Rating vs Viewers (Colored by Season)")

alt.Chart(...)

In [23]:
# Scatter colored by era (3 groups)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

alt.Chart(df).mark_circle(size=50, opacity=0.6).encode(
    x=alt.X("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
    y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
    color=alt.Color(
        "era:N",
        scale=alt.Scale(domain=era_domain, range=era_colors),
        title="Era",
    ),
    tooltip=[
        "title:N",
        "season:O",
        "era:N",
        "imdb_rating:Q",
        "us_viewers_in_millions:Q",
    ],
).properties(width=500, height=400, title="Rating vs Viewers (Colored by Era)")

alt.Chart(...)

In [24]:
# Scatter + single regression line
q3d_scatter = (
    alt.Chart(df)
    .mark_circle(size=40, opacity=0.4)
    .encode(
        x=alt.X("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("#4c78a8"),
        tooltip=["title:N", "season:O", "imdb_rating:Q", "us_viewers_in_millions:Q"],
    )
)

q3d_reg = (
    q3d_scatter.transform_regression("imdb_rating", "us_viewers_in_millions")
    .mark_line(color="#e45756", strokeWidth=2)
    .encode(color=alt.value("#e45756"))
)

(q3d_scatter + q3d_reg).properties(
    width=500, height=400, title="Rating vs Viewers (with Regression Line)"
)

alt.LayerChart(...)

In [25]:
# Scatter + regression per era
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]
era_scale = alt.Scale(domain=era_domain, range=era_colors)

q3e_scatter = (
    alt.Chart(df)
    .mark_circle(size=50, opacity=0.5)
    .encode(
        x=alt.X("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color("era:N", scale=era_scale, title="Era"),
        tooltip=[
            "title:N",
            "season:O",
            "era:N",
            "imdb_rating:Q",
            "us_viewers_in_millions:Q",
        ],
    )
)

q3e_reg = (
    q3e_scatter.transform_regression(
        "imdb_rating", "us_viewers_in_millions", groupby=["era"]
    )
    .mark_line(strokeWidth=2.5)
    .encode(color=alt.Color("era:N", scale=era_scale, title="Era"))
)

(q3e_scatter + q3e_reg).properties(
    width=500, height=400, title="Rating vs Viewers (Regression per Era)"
)

alt.LayerChart(...)

In [26]:
# Scatter + regression per era (flipped axes: viewers on x, rating on y)
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]
era_scale = alt.Scale(domain=era_domain, range=era_colors)

q3f_scatter = (
    alt.Chart(df)
    .mark_circle(size=50, opacity=0.5)
    .encode(
        x=alt.X("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        y=alt.Y("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
        color=alt.Color("era:N", scale=era_scale, title="Era"),
        tooltip=[
            "title:N",
            "season:O",
            "era:N",
            "imdb_rating:Q",
            "us_viewers_in_millions:Q",
        ],
    )
)

q3f_reg = (
    q3f_scatter.transform_regression(
        "us_viewers_in_millions", "imdb_rating", groupby=["era"]
    )
    .mark_line(strokeWidth=2.5)
    .encode(color=alt.Color("era:N", scale=era_scale, title="Era"))
)

(q3f_scatter).properties(
    width=500, height=400, title="Viewers vs Rating (Regression per Era, flipped axes)"
)

alt.Chart(...)

## Are the number of viewers related to the weekday they were aired?

First we analyze the weekday distribution to understand which days are meaningful, then compare Thursday vs Sunday.


In [27]:
# Weekday distribution per season
weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
ct = pd.crosstab(df["season"], df["weekday"]).reindex(
    columns=weekday_order, fill_value=0
)
ct["Total"] = ct.sum(axis=1)
print(ct.to_string())

print("\nSummary:")
for day in weekday_order:
    n = (df["weekday"] == day).sum()
    if n > 0:
        seasons = sorted(
            int(s) for s in df.loc[df["weekday"] == day, "season"].unique()
        )
        print(f"  {day}: {n} episodes (seasons {seasons})")

weekday  Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday  Total
season                                                                        
1             0        0          0         0       0         0      13     13
2             0        0          0        22       0         0       0     22
3             0        0          0        24       0         0       0     24
4             0        1          0        21       0         0       0     22
5             0        0          0        22       0         0       0     22
6             0        0          0         0       0         0      25     25
7             0        0          0         0       0         0      25     25
8             0        0          0         0       1         0      24     25
9             0        0          0         0       0         0      25     25
10            0        0          0         0       0         0      23     23
11            0        0          0         0       

In [28]:
# Non-Sunday episodes
non_sunday = df[df["weekday"] != "Sunday"].sort_values("original_air_date")
print(f"Total non-Sunday episodes: {len(non_sunday)}\n")
print(
    non_sunday[
        [
            "title",
            "season",
            "number_in_season",
            "weekday",
            "original_air_date",
            "us_viewers_in_millions",
            "imdb_rating",
        ]
    ].to_string()
)

Total non-Sunday episodes: 94

                                                                            title  season  number_in_season    weekday original_air_date  us_viewers_in_millions  imdb_rating
13                                                               Bart Gets an "F"       2                 1   Thursday        1990-10-11                    33.6          8.2
14                                                            Simpson and Delilah       2                 2   Thursday        1990-10-18                    29.9          8.3
15                                                            Treehouse of Horror       2                 3   Thursday        1990-10-25                    27.4          8.2
16                          Two Cars in Every Garage and Three Eyes on Every Fish       2                 4   Thursday        1990-11-01                    26.1          8.1
17                                                                  Dancin' Homer       2          

**Key observation:** Thursday episodes come exclusively from seasons 2-5 (the show's peak viewership era with 20-30M viewers). Sunday episodes span season 1 and seasons 6-27. The 5 episodes on other days (Tue, Wed, Fri) are too few to analyze. Any apparent weekday effect is fully confounded with the temporal trend: no season aired on both Thursday and Sunday, so a fair comparison is structurally impossible.


In [29]:
# Filter to the two meaningful weekdays
df_thu_sun = df[df["weekday"].isin(["Thursday", "Sunday"])].copy()
print(f"Thursday vs Sunday: {len(df_thu_sun)} episodes")
print(df_thu_sun["weekday"].value_counts())

Thursday vs Sunday: 591 episodes
weekday
Sunday      502
Thursday     89
Name: count, dtype: int64


In [30]:
# Box plot Thursday vs Sunday
# Thursday = seasons 2-5 (Golden Age), Sunday = seasons 1+6-27 (spans all eras)
thu_sun_labels = ["Thursday\n(Golden Age)", "Sunday\n(Middle + Later)"]

df_thu_sun["weekday_label"] = df_thu_sun["weekday"].map(
    {
        "Thursday": thu_sun_labels[0],
        "Sunday": thu_sun_labels[1],
    }
)

alt.Chart(df_thu_sun).mark_boxplot(size=60).encode(
    x=alt.X(
        "weekday_label:N",
        sort=thu_sun_labels,
        title="Day of the Week",
        axis=alt.Axis(labelAngle=0),
    ),
    y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
    color=alt.Color(
        "weekday_label:N",
        sort=thu_sun_labels,
        legend=None,
        scale=alt.Scale(domain=thu_sun_labels, range=["#4c78a8", "#e45756"]),
    ),
).properties(width=400, height=400, title="US Viewers: Thursday vs Sunday")

alt.Chart(...)

In [31]:
# Strip + box plot combined (Thu vs Sun), dots colored by era
np.random.seed(42)
df_thu_sun["jitter_offset"] = np.random.uniform(-0.2, 0.2, len(df_thu_sun))

era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

q4e_box = (
    alt.Chart(df_thu_sun)
    .mark_boxplot(size=60, opacity=0.2)
    .encode(
        x=alt.X(
            "weekday_label:N",
            sort=thu_sun_labels,
            title="Day of the Week",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.value("gray"),
    )
)

q4e_dots = (
    alt.Chart(df_thu_sun)
    .mark_circle(size=30, opacity=0.6)
    .encode(
        x=alt.X(
            "weekday_label:N",
            sort=thu_sun_labels,
            title="Day of the Week",
            axis=alt.Axis(labelAngle=0),
        ),
        xOffset=alt.XOffset(
            "jitter_offset:Q",
            scale=alt.Scale(domain=[-0.4, 0.4]),
        ),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
        tooltip=[
            "title:N",
            "season:O",
            "era:N",
            "weekday:N",
            "us_viewers_in_millions:Q",
        ],
    )
)

(q4e_box + q4e_dots).properties(
    width=400, height=400, title="US Viewers: Thursday vs Sunday (Strip + Box)"
)

alt.LayerChart(...)

## Do the seasons' number of viewers present any relevant pattern?

We focus on within-season patterns: do premieres and finales attract more viewers than mid-season episodes? Has this pattern changed across eras? We also keep the heatmap for a complete overview.


In [32]:
# Heatmap (season x episode position)
alt.Chart(df).mark_rect().encode(
    x=alt.X(
        "number_in_season:O",
        title="Episode in Season",
        axis=alt.Axis(labelAngle=0),
    ),
    y=alt.Y("season:O", title="Season", sort="descending"),
    color=alt.Color(
        "us_viewers_in_millions:Q",
        scale=alt.Scale(scheme="blues"),
        title="Viewers (M)",
    ),
    tooltip=["title:N", "season:O", "number_in_season:O", "us_viewers_in_millions:Q"],
).properties(
    width=600, height=500, title="Viewership Heatmap (Season x Episode Position)"
)

alt.Chart(...)

In [33]:
# Overlaid line chart (all seasons)
alt.Chart(df).mark_line(opacity=0.4, strokeWidth=1).encode(
    x=alt.X("number_in_season:Q", title="Episode Position in Season"),
    y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
    color=alt.Color("season:Q", scale=alt.Scale(scheme="turbo"), title="Season"),
    detail="season:N",
    tooltip=["season:O", "number_in_season:O", "us_viewers_in_millions:Q"],
).properties(
    width=600,
    height=400,
    title="Within-Season Viewership (All Seasons Overlaid)",
)

alt.Chart(...)

In [34]:
# Mean viewership by episode position (aggregated across all seasons)
pos_stats = (
    df.groupby("number_in_season")["us_viewers_in_millions"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_stats = pos_stats[pos_stats["count"] >= 10]

overall_mean = df["us_viewers_in_millions"].mean()

q5_agg_line = (
    alt.Chart(pos_stats)
    .mark_line(point=True, strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y("mean:Q", title="Mean US Viewers (millions)"),
        color=alt.value("#4c78a8"),
        tooltip=[
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Mean Viewers (M)", format=".2f"),
            alt.Tooltip("count:Q", title="Seasons with this position"),
        ],
    )
)

q5_agg_ref = (
    alt.Chart(pd.DataFrame({"y": [overall_mean]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5_agg_line + q5_agg_ref).properties(
    width=600,
    height=350,
    title="Mean Viewership by Episode Position (Aggregated Across Seasons)",
)

alt.LayerChart(...)

In [35]:
# Premiere vs Finale vs Season Average
season_stats = (
    df.groupby("season")
    .agg(
        premiere=("us_viewers_in_millions", "first"),
        finale=("us_viewers_in_millions", "last"),
        average=("us_viewers_in_millions", "mean"),
    )
    .reset_index()
)

# Reshape to long format for plotting
season_long = season_stats.melt(
    id_vars="season",
    value_vars=["premiere", "finale", "average"],
    var_name="type",
    value_name="us_viewers_in_millions",
)

type_domain = ["premiere", "average", "finale"]
type_colors = ["#4c78a8", "#72b7b2", "#e45756"]

q5_lines = (
    alt.Chart(season_long)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "type:N",
            scale=alt.Scale(domain=type_domain, range=type_colors),
            title="Episode Type",
        ),
    )
)

q5_points = (
    alt.Chart(season_long)
    .mark_point(filled=True, size=40)
    .encode(
        x=alt.X("season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q"),
        color=alt.Color(
            "type:N",
            scale=alt.Scale(domain=type_domain, range=type_colors),
            title="Episode Type",
        ),
        tooltip=[
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("type:N", title="Type"),
            alt.Tooltip("us_viewers_in_millions:Q", title="Viewers (M)", format=".2f"),
        ],
    )
)

# Overall series mean
overall_mean = df["us_viewers_in_millions"].mean()
q5_mean_rule = (
    alt.Chart(pd.DataFrame({"y": [overall_mean]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5_lines + q5_points + q5_mean_rule).properties(
    width=700, height=350, title="Premiere vs Finale vs Season Average Viewership"
)

alt.LayerChart(...)

In [36]:
# Season quarter viewership (normalized by season mean)
def assign_quarter(row):
    total_eps = df[df["season"] == row["season"]]["number_in_season"].max()
    pos = row["number_in_season"]
    if pos <= total_eps * 0.25:
        return "Q1 (Start)"
    elif pos <= total_eps * 0.5:
        return "Q2"
    elif pos <= total_eps * 0.75:
        return "Q3"
    else:
        return "Q4 (End)"


df["season_quarter"] = df.apply(assign_quarter, axis=1)

# Normalize: each episode's viewers / season mean
season_avg = df.groupby("season")["us_viewers_in_millions"].transform("mean")
df["viewers_normalized"] = df["us_viewers_in_millions"] / season_avg

quarter_norm = (
    df.groupby(["season", "season_quarter"])["viewers_normalized"].mean().reset_index()
)

quarter_order = ["Q1 (Start)", "Q2", "Q3", "Q4 (End)"]
quarter_colors = ["#4c78a8", "#72b7b2", "#f58518", "#e45756"]

q5q_lines = (
    alt.Chart(quarter_norm)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "viewers_normalized:Q",
            title="Viewers (relative to season average)",
            axis=alt.Axis(format=".0%"),
        ),
        color=alt.Color(
            "season_quarter:N",
            scale=alt.Scale(domain=quarter_order, range=quarter_colors),
            title="Season Quarter",
        ),
    )
)

q5q_points = (
    alt.Chart(quarter_norm)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("viewers_normalized:Q"),
        color=alt.Color(
            "season_quarter:N",
            scale=alt.Scale(domain=quarter_order, range=quarter_colors),
            title="Season Quarter",
        ),
        tooltip=[
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("season_quarter:N", title="Quarter"),
            alt.Tooltip("viewers_normalized:Q", title="Relative Viewers", format=".1%"),
        ],
    )
)

# Reference line at 100% (season average)
q5q_ref = (
    alt.Chart(pd.DataFrame({"y": [1.0]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5q_lines + q5q_points + q5q_ref).properties(
    width=700,
    height=350,
    title="Within-Season Viewership Pattern (Normalized by Season Average)",
)

alt.LayerChart(...)

In [37]:
# Season thirds viewership (normalized by season mean)
def assign_third(row):
    total_eps = df[df["season"] == row["season"]]["number_in_season"].max()
    pos = row["number_in_season"]
    if pos <= total_eps / 3:
        return "Start (1/3)"
    elif pos <= total_eps * 2 / 3:
        return "Middle (1/3)"
    else:
        return "End (1/3)"


df["season_third"] = df.apply(assign_third, axis=1)

third_norm = (
    df.groupby(["season", "season_third"])["viewers_normalized"].mean().reset_index()
)

third_order = ["Start (1/3)", "Middle (1/3)", "End (1/3)"]
third_colors = ["#4c78a8", "#72b7b2", "#e45756"]

q5t_lines = (
    alt.Chart(third_norm)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y(
            "viewers_normalized:Q",
            title="Viewers (relative to season average)",
            axis=alt.Axis(format=".0%"),
        ),
        color=alt.Color(
            "season_third:N",
            scale=alt.Scale(domain=third_order, range=third_colors),
            title="Season Third",
        ),
    )
)

q5t_points = (
    alt.Chart(third_norm)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("viewers_normalized:Q"),
        color=alt.Color(
            "season_third:N",
            scale=alt.Scale(domain=third_order, range=third_colors),
            title="Season Third",
        ),
        tooltip=[
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("season_third:N", title="Third"),
            alt.Tooltip("viewers_normalized:Q", title="Relative Viewers", format=".1%"),
        ],
    )
)

q5t_ref = (
    alt.Chart(pd.DataFrame({"y": [1.0]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5t_lines + q5t_points + q5t_ref).properties(
    width=700, height=350, title="Within-Season Viewership Pattern (Normalized, Thirds)"
)

alt.LayerChart(...)

In [38]:
# Season quarter viewership (NOT normalized, absolute values)
quarter_abs = (
    df.groupby(["season", "season_quarter"])["us_viewers_in_millions"]
    .mean()
    .reset_index()
)

q5q_abs_lines = (
    alt.Chart(quarter_abs)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
        color=alt.Color(
            "season_quarter:N",
            scale=alt.Scale(domain=quarter_order, range=quarter_colors),
            title="Season Quarter",
        ),
    )
)

q5q_abs_points = (
    alt.Chart(quarter_abs)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("us_viewers_in_millions:Q"),
        color=alt.Color(
            "season_quarter:N",
            scale=alt.Scale(domain=quarter_order, range=quarter_colors),
            title="Season Quarter",
        ),
        tooltip=[
            alt.Tooltip("season:O", title="Season"),
            alt.Tooltip("season_quarter:N", title="Quarter"),
            alt.Tooltip("us_viewers_in_millions:Q", title="Viewers (M)", format=".2f"),
        ],
    )
)

(q5q_abs_lines + q5q_abs_points).properties(
    width=700, height=350, title="Within-Season Viewership Pattern (Absolute Values)"
)

alt.LayerChart(...)

In [39]:
# Mean normalized viewership by episode position, split by era
era_domain = ["Golden Age (S1-8)", "Middle (S9-18)", "Later (S19-27)"]
era_colors = ["#4c78a8", "#f58518", "#e45756"]

pos_era_norm = (
    df.groupby(["era", "number_in_season"])["viewers_normalized"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_era_norm = pos_era_norm[pos_era_norm["count"] >= 3]

q5f_lines = (
    alt.Chart(pos_era_norm)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y(
            "mean:Q",
            title="Viewers (relative to season average)",
            axis=alt.Axis(format=".0%"),
        ),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
        tooltip=[
            alt.Tooltip("era:N", title="Era"),
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Relative Viewers", format=".1%"),
            alt.Tooltip("count:Q", title="Seasons"),
        ],
    )
)

q5f_points = (
    alt.Chart(pos_era_norm)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("number_in_season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean:Q"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
    )
)

q5f_ref = (
    alt.Chart(pd.DataFrame({"y": [1.0]}))
    .mark_rule(color="black", strokeDash=[4, 4], opacity=0.5)
    .encode(y="y:Q")
)

(q5f_lines + q5f_points + q5f_ref).properties(
    width=700,
    height=350,
    title="Within-Season Viewership by Episode Position (Normalized, Split by Era)",
)

alt.LayerChart(...)

In [40]:
# Mean viewership by episode position, split by era (NOT normalized, absolute values)
pos_era_abs = (
    df.groupby(["era", "number_in_season"])["us_viewers_in_millions"]
    .agg(["mean", "count"])
    .reset_index()
)
pos_era_abs = pos_era_abs[pos_era_abs["count"] >= 3]

q5f_abs_lines = (
    alt.Chart(pos_era_abs)
    .mark_line(strokeWidth=2)
    .encode(
        x=alt.X(
            "number_in_season:O",
            title="Episode Position in Season",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y("mean:Q", title="Mean US Viewers (millions)"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
        tooltip=[
            alt.Tooltip("era:N", title="Era"),
            alt.Tooltip("number_in_season:O", title="Episode"),
            alt.Tooltip("mean:Q", title="Mean Viewers (M)", format=".2f"),
            alt.Tooltip("count:Q", title="Seasons"),
        ],
    )
)

q5f_abs_points = (
    alt.Chart(pos_era_abs)
    .mark_point(filled=True, size=35)
    .encode(
        x=alt.X("number_in_season:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean:Q"),
        color=alt.Color(
            "era:N",
            scale=alt.Scale(domain=era_domain, range=era_colors),
            title="Era",
        ),
    )
)

(q5f_abs_lines + q5f_abs_points).properties(
    width=700,
    height=350,
    title="Within-Season Viewership by Episode Position (Absolute, Split by Era)",
)

alt.LayerChart(...)